In [ ]:
# Kiểm tra Cuda có được hỗ trợ không
import torch
print(f"CUDA khả dụng: {torch.cuda.is_available()}")
print(f"Thiết bị: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

In [ ]:
import openslide
import cv2
import numpy as np
import torch
import torchvision.models as models
import torchvision.transforms as transforms
import staintools
from PIL import Image

class WSIPreprocessor:
    def __init__(self, svs_path, patch_size=256):
        self.slide = openslide.OpenSlide(svs_path)
        self.patch_size = patch_size
        self.dimensions = self.slide.dimensions # Kích thước ở level 0 (cao nhất)
        
        # Setup ResNet50 cho Feature Extraction (Bước 4)
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
        self.model.fc = torch.nn.Identity() # Bỏ lớp classification cuối để lấy vector đặc trưng
        self.model.eval().to(self.device)
        
        self.transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])

        # Bộ chuẩn hóa màu Macenko (Bước 5)
        self.normalizer = staintools.StainNormalizer(method='macenko')
        # Tải một ảnh tham chiếu (target) có màu H&E chuẩn để train normalizer
        # Ở đây giả lập, trong thực tế bạn cần 1 patch chuẩn.
        # target_img = staintools.read_image("standard_HE_patch.png")
        # self.normalizer.fit(target_img)

    def step1_tissue_segmentation(self, level=2):
        """Bước 1: Tách nền bằng cách đọc ảnh ở độ phân giải thấp (level 2 hoặc 3)"""
        # Đọc ảnh thu nhỏ để xử lý cho nhanh
        downsample = self.slide.level_downsamples[level]
        w, h = self.slide.level_dimensions[level]
        thumbnail = self.slide.read_region((0, 0), level, (w, h)).convert("RGB")
        thumbnail_np = np.array(thumbnail)

        # Chuyển sang không gian màu HSV và áp dụng Otsu thresholding
        hsv = cv2.cvtColor(thumbnail_np, cv2.COLOR_RGB2HSV)
        saturation = hsv[:, :, 1]
        _, mask = cv2.threshold(saturation, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        
        return mask, downsample

    def step3_quality_control(self, patch_img):
        """Bước 3: Lọc nhiễu (Kiểm tra lượng mô trắng và độ mờ)"""
        patch_np = np.array(patch_img)
        gray = cv2.cvtColor(patch_np, cv2.COLOR_RGB2GRAY)
        
        # Lọc ảnh rỗng/trắng (Nền chiếm > 70%)
        white_pixels = np.sum(gray > 210)
        if white_pixels / (self.patch_size * self.patch_size) > 0.7:
            return False
            
        # Lọc ảnh mờ (Laplacian variance)
        laplacian_var = cv2.Laplacian(gray, cv2.CV_64F).var()
        if laplacian_var < 50: # Ngưỡng này có thể tinh chỉnh
            return False
            
        return True

    def process_slide(self):
        """Thực thi toàn bộ pipeline"""
        print("1. Đang tách nền (Tissue Segmentation)...")
        tissue_mask, downsample = self.step1_tissue_segmentation(level=2)
        
        features_list = []
        coords_list = []

        print("2. Đang cắt ảnh và xử lý từng patch...")
        w, h = self.dimensions
        
        # Duyệt qua tọa độ ảnh gốc (Level 0) với bước nhảy là patch_size
        for y in range(0, h, self.patch_size):
            for x in range(0, w, self.patch_size):
                
                # Ánh xạ tọa độ level 0 xuống level của mask để kiểm tra có nằm trong vùng mô không
                mask_x = int(x / downsample)
                mask_y = int(y / downsample)
                
                # Tránh lỗi out of bounds
                if mask_y >= tissue_mask.shape[0] or mask_x >= tissue_mask.shape[1]:
                    continue
                    
                # Chỉ cắt nếu tâm của patch nằm trong vùng có mô (mask > 0)
                if tissue_mask[mask_y, mask_x] > 0:
                    
                    # Cắt ảnh (Patching/Tiling)
                    patch = self.slide.read_region((x, y), 0, (self.patch_size, self.patch_size)).convert("RGB")
                    
                    # Lọc nhiễu (Quality Control)
                    if not self.step3_quality_control(patch):
                        continue
                        
                    # Chuẩn hóa màu sắc (Stain Normalization)
                    # patch_np = np.array(patch)
                    # try:
                    #    patch_np = self.normalizer.transform(patch_np)
                    # except:
                    #    continue # Bỏ qua nếu lỗi chuẩn hóa
                    # patch = Image.fromarray(patch_np)

                    # Trích xuất đặc trưng (Feature Extraction)
                    input_tensor = self.transform(patch).unsqueeze(0).to(self.device)
                    with torch.no_grad():
                        vector = self.model(input_tensor)
                    
                    features_list.append(vector.cpu().numpy().squeeze())
                    coords_list.append((x, y))

        print(f"Đã xử lý xong! Tổng số patches hợp lệ: {len(coords_list)}")
        return np.array(features_list), np.array(coords_list)

    def extract_features_batch(self, svs_path, valid_coords, batch_size=64, num_workers=4):
        """
        Thực hiện Batch Feature Extraction qua GPU
        - valid_coords: Danh sách các tuple (x, y) đã pass qua bước lọc nền và lọc nhiễu.
        """
        print(f"Bắt đầu trích xuất đặc trưng cho {len(valid_coords)} patches với batch_size={batch_size}...")
        
        # 1. Khởi tạo Dataset và DataLoader
        dataset = WSIPatchDataset(svs_path, valid_coords, self.patch_size, self.transform)
        
        # num_workers giúp CPU cắt ảnh song song trong lúc GPU đang bận tính toán batch trước đó
        dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)
        
        features_list = []
        final_coords = []
        
        self.model.eval()
        
        # Bật Mixed Precision (Tùy chọn) để tăng tốc độ x2 trên các dòng GPU đời mới (RTX, T4, A100...)
        scaler = torch.cuda.amp.autocast() if torch.cuda.is_available() else contextlib.nullcontext()

        with torch.no_grad(): # Tắt tính toán đạo hàm để tiết kiệm RAM
            for batch_patches, batch_coords in tqdm(dataloader, desc="Extracting"):
                # Đẩy batch ảnh lên GPU
                batch_patches = batch_patches.to(self.device)
                
                # 2. Chạy qua model (Forward pass)
                with scaler:
                    features = self.model(batch_patches)
                
                # 3. Kéo kết quả về CPU và lưu tạm
                # Ép kiểu về float16/float32 tùy nhu cầu lưu trữ
                features_list.append(features.cpu().numpy()) 
                final_coords.append(batch_coords.numpy())
                
        # 4. Gộp (stack) tất cả các batch lại thành một ma trận duy nhất
        all_features = np.vstack(features_list)
        all_coords = np.vstack(final_coords)
        
        return all_features, all_coords
# Cách sử dụng:
    

In [ ]:
preprocessor = WSIPreprocessor("../data/raw_wsi/TCGA-06-0210-01Z-00-DX3.4974dd3d-13bb-4200-ab3d-3dfd1dcffbb9.svs", patch_size=64)
features, coordinates = preprocessor.process_slide()
np.save("features.npy", features)
np.save("coords.npy", coordinates)